In [6]:
!pip install requests beautifulsoup4

In [7]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import csv

url = "https://books.toscrape.com/"

try:
    # 1. 페이지 요청
    response = requests.get(url, timeout=10)
    response.raise_for_status()  # 4xx, 5xx 오류 확인

    # 2. HTML 파싱
    soup = BeautifulSoup(response.text, "html.parser")

    # 3. 책 하나를 감싸는 article 요소 모두 찾기
    articles = soup.select("article.product_pod")

    books = []

    for article in articles:
        # 제목과 링크가 들어 있는 a 태그
        title_a = article.select_one("h3 a")

        # 가격
        price_p = article.select_one("p.price_color")

        # 제목
        if title_a:
            # title 속성이 있으면 사용하고,
            # 없으면 a 태그의 글자를 사용
            title = title_a.get("title") or title_a.get_text(strip=True)

            # 상대 URL → 절대 URL
            href = title_a.get("href", "")
            link = urljoin(url, href) if href else ""
        else:
            title = ""
            link = ""

        # 가격 요소가 없으면 빈칸
        price = price_p.get_text(strip=True) if price_p else ""

        books.append({
            "제목": title,
            "가격": price,
            "상세 페이지 링크": link
        })

    # 4. CSV 저장
    with open("books.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["제목", "가격", "상세 페이지 링크"]
        )

        writer.writeheader()
        writer.writerows(books)

    # 5. 수집 결과 확인
    print(f"HTTP 상태 코드: {response.status_code}")
    print(f"수집 건수: {len(books)}건")
    print("books.csv 저장 완료")

except requests.RequestException as e:
    print(f"HTTP 요청 오류: {e}")

HTTP 상태 코드: 200
수집 건수: 20건
books.csv 저장 완료
